# 📊 NovaMart Data Profiling & Exploratory Quality Analysis

**Project:** NovaMart Financial Analytics & Revenue Forecasting  
**Task:** Task 4 — Exploratory Data Profiling  
**Target Dataset:** `data/raw/square_item_sales_detail_24mo.csv` (970,838 rows)  


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load raw dataset
sales_path = '../data/raw/square_item_sales_detail_24mo.csv'
df = pd.read_csv(sales_path, low_memory=False)
df['parsed_date'] = pd.to_datetime(df['Date'])
df['cogs'] = df['Qty'] * df['Unit Cost']
df['YearMonth'] = df['parsed_date'].dt.to_period('M')

print(f"Loaded dataset with {len(df):,} rows and {len(df.columns)} columns.")


## 1. Basic Data Profile & Missingness Summary


In [2]:
profile_table = []
for col in df.columns:
    profile_table.append({
        'Column': col,
        'Data Type': str(df[col].dtype),
        'Non-Null Count': df[col].notnull().sum(),
        'Null Count': df[col].isnull().sum(),
        'Null %': f"{(df[col].isnull().sum() / len(df))*100:.2f}%",
        'Unique Count': df[col].nunique(dropna=False)
    })
pd.DataFrame(profile_table)


## 2. Business Key Uniqueness Check


In [3]:
bk_cols = ['Transaction ID', 'SKU', 'Event Type']
dups = df.duplicated(subset=bk_cols).sum()
print(f"Business Key (Transaction ID + SKU + Event Type) Duplicate Count: {dups}")
assert dups == 0, 'Business key is not unique!'


## 3. Financial Formula Reconciliations


In [4]:
gross_reconcile = ((df['Gross Sales'] - (df['Qty'] * df['Unit Price'])).abs() <= 0.01).all()
net_reconcile = ((df['Net Sales'] - (df['Gross Sales'] + df['Discounts'])).abs() <= 0.01).all()
profit_reconcile = ((df['Gross Profit'] - (df['Net Sales'] - df['cogs'])).abs() <= 0.01).all()

print(f"Gross Sales Formula Reconciliation: {gross_reconcile}")
print(f"Net Sales Formula Reconciliation:   {net_reconcile}")
print(f"Gross Profit Formula Reconciliation: {profit_reconcile}")


## 4. Monthly Financial Trend & Visualizations


In [5]:
monthly = df.groupby('YearMonth').agg(
    Net_Sales=('Net Sales', 'sum'),
    Gross_Profit=('Gross Profit', 'sum'),
    Transactions=('Transaction ID', 'nunique')
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
months_str = monthly['YearMonth'].astype(str)
ax1.plot(months_str, monthly['Net_Sales'] / 1e6, marker='o', color='#1f77b4', linewidth=2, label='Net Sales ($M)')
ax1.plot(months_str, monthly['Gross_Profit'] / 1e6, marker='s', color='#2ca02c', linewidth=2, label='Gross Profit ($M)')
ax1.set_title('NovaMart Monthly Net Sales & Gross Profit (2024–2025)', fontsize=14, fontweight='bold')
ax1.set_ylabel('USD ($ Millions)')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()
